# Kaggle Training + Evaluation (EfficientNet-B4)

This notebook runs one clean pipeline:
1. Setup project in `/kaggle/working`
2. Build splits
3. Train EfficientNet-B4
4. Evaluate and export metrics + model


In [ ]:
from pathlib import Path
import os
import shutil

KAGGLE_INPUT = Path('/kaggle/input')
PROJECT_ROOT = Path('/kaggle/working/EYE_HEART_CONNECTION')
ARTIFACT_DIR = Path('/kaggle/working/eye_heart_artifacts')

candidates = list(KAGGLE_INPUT.rglob('full_df.csv'))
if not candidates:
    raise FileNotFoundError('Could not find full_df.csv under /kaggle/input.')

source_root = candidates[0].parent
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(source_root, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
(ARTIFACT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(ARTIFACT_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(ARTIFACT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

print('Working directory:', Path.cwd())


In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -e .[dev]


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!python -m datasets.build_patient_df --config configs/data.yaml
!python -m datasets.data_quality --data-config configs/data.yaml


In [ ]:
from pathlib import Path
import yaml

EPOCHS = 15  # change as needed

with Path('configs/train.yaml').open('r', encoding='utf-8') as f:
    train_cfg = yaml.safe_load(f)

train_cfg['epochs'] = EPOCHS
train_cfg['experiment']['name'] = f'kaggle_effb4_{EPOCHS}ep'

train_cfg_path = Path('configs/train_kaggle.yaml')
with train_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False)

print('Train config written:', train_cfg_path)


In [ ]:
!python -m training.train --config configs/train_kaggle.yaml --data-config configs/data.yaml --model-config configs/model.yaml


In [ ]:
!python -m evaluation.run --config configs/eval.yaml --data-config configs/data.yaml --model-config configs/model.yaml --ckpt experiments/latest/best.pt


In [ ]:
from pathlib import Path
import shutil

copy_items = [
    (Path('experiments/latest/best.pt'), ARTIFACT_DIR / 'models' / 'efficientnet_b4_best.pt'),
    (Path('data/processed/eval_report.json'), ARTIFACT_DIR / 'metrics' / 'eval_report.json'),
    (Path('data/processed/eval_predictions.csv'), ARTIFACT_DIR / 'metrics' / 'eval_predictions.csv'),
    (Path('data/processed/thresholds.json'), ARTIFACT_DIR / 'metrics' / 'thresholds.json'),
]
for src, dst in copy_items:
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

for run_metrics in Path('experiments').glob('kaggle_effb4_*/*metrics.csv'):
    dst = ARTIFACT_DIR / 'logs' / run_metrics.parent.name
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(run_metrics, dst / 'metrics.csv')

for p in sorted(ARTIFACT_DIR.rglob('*')):
    if p.is_file():
        print(p)


In [ ]:
!tar -czf /kaggle/working/eye_heart_artifacts.tar.gz -C /kaggle/working eye_heart_artifacts
print('Archive created: /kaggle/working/eye_heart_artifacts.tar.gz')
